# Stage 3: TRL DPOTrainer + LoRA persona tuning (SmolLM2-360M-Instruct)

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/muttaqi123/small-model-alignment.git"
COMMIT = "5713785"
WORK = Path("/kaggle/working/small-model-alignment")
os.chdir("/kaggle/working")
if not WORK.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(WORK)], check=True)
subprocess.run(["git", "-C", str(WORK), "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", str(WORK), "checkout", "-q", COMMIT], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = str(WORK)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


def run(cmd, env=None):
    """Run a shell command, stream its output, and fail the notebook on error."""
    print(f"$ {cmd}", flush=True)
    merged = {**os.environ, **(env or {})}
    proc = subprocess.Popen(cmd, shell=True, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")


def start(cmd, log_path, env=None):
    """Start a background command (e.g. generation on the second GPU)."""
    print(f"$ {cmd} > {log_path} &", flush=True)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    handle = open(log_path, "w")
    merged = {**os.environ, **(env or {})}
    return subprocess.Popen(cmd, shell=True, env=merged, stdout=handle, stderr=subprocess.STDOUT, text=True), log_path


def wait(job):
    proc, log_path = job
    code = proc.wait()
    print(Path(log_path).read_text()[-3000:])
    if code != 0:
        raise RuntimeError(f"background job failed: {log_path}")


def archive(zip_path, items, exclude=("trainer", "best", "final", "adapter")):
    """Zip small artifacts (metrics, configs, generations, plots); never weights."""
    import zipfile
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in items:
            p = Path(item)
            files = [p] if p.is_file() else sorted(f for f in p.rglob("*") if f.is_file())
            for f in files:
                if any(part in exclude for part in f.parts) or f.suffix in (".safetensors", ".bin", ".pt"):
                    continue
                zf.write(f, str(f))
    print(f"{zip_path}: {Path(zip_path).stat().st_size / 1e6:.1f} MB")


def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"no input matches {pattern}")
    print(f"found {pattern} -> {hits[0]}")
    return hits[0]

In [ ]:
run("nvidia-smi")
run("python -m pip uninstall -y -q torchao || true")
run('python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "trl>=0.19,<0.24" "peft>=0.15,<0.18"')
run("python -m pip install -q -e . --no-deps")
run('python -c "import torch, transformers, datasets; print(\'torch\', torch.__version__, \'| transformers\', transformers.__version__, \'| datasets\', datasets.__version__, \'| cuda\', torch.version.cuda, \'| gpus\', torch.cuda.device_count())"')
run("python -m pytest -q tests/")

In [ ]:
run("python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo", env={"CUDA_VISIBLE_DEVICES": "0"})
print(open("outputs/lora_dpo/sweep_summary.json").read()[:6000])

In [ ]:
base = "HuggingFaceTB/SmolLM2-360M-Instruct"
run(f"python -m alignment.generate --batch_size 16 --max_new_tokens 256 --repetition_penalty 1.1 --model_path {base} --prompt_format chat_template --eval_path eval_sets/persona_style_eval_40.json "
    "--output_path outputs/generations/persona_base.json --generator smollm2-360m-instruct")
for name in ("r8_attn_lr5e-5", "r16_all_lr5e-5", "r16_all_lr2e-4"):
    run(f"python -m alignment.generate --batch_size 16 --max_new_tokens 256 --repetition_penalty 1.1 --model_path {base} --adapter_path outputs/lora_dpo/{name}/adapter --prompt_format chat_template "
        f"--eval_path eval_sets/persona_style_eval_40.json --output_path outputs/generations/persona_lora_{name}.json "
        f"--generator smollm2-360m-lora-dpo-{name}")
for name in ("persona_base", "persona_lora_r16_all_lr5e-5", "persona_lora_r16_all_lr2e-4"):
    print("#" * 30, name)
    for r in json.load(open(f"outputs/generations/{name}.json"))[::8]:
        print("=" * 80, "\n", r["instruction"], "\n---\n", r["output"][:700])

In [ ]:
run("python scripts/plot_metrics.py --lora outputs/lora_dpo --out_dir outputs/plots_lora")
archive("lora_dpo_run_outputs.zip", ["outputs/lora_dpo", "outputs/generations", "outputs/plots_lora"])
run("du -sh outputs/*")